In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.__version__)

True
2.11.0+cu128


In [ ]:
!pip install -q flask flask-cors pyngrok

In [ ]:
!pip install flask-cors

In [ ]:
!pip install -q "numpy>=2.0,<2.2" "scipy>=1.14" "diffusers==0.31.0" "peft==0.13.2" "transformers==4.46.3" "pillow==10.4.0" accelerate safetensors rembg onnxruntime flask flask-cors pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub

In [ ]:
token = "3IuTYphCzh7b5C45QsHeLqqnRF2_5gVfHjeKEmHecPZYTG5ct"
print(len(token))  # 토큰 길이가 0이면 아직 안 들어간 것

49


In [ ]:
from flask import Flask, request, send_file
from flask_cors import CORS
from pyngrok import ngrok
import io, torch
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from rembg import remove, new_session

app = Flask(__name__)
CORS(app)

# LoRA 없이 base 모델만 (특정 개체에 묶이지 않도록)
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
bg_session = new_session("u2net")

# 한글 입력을 영어 프롬프트로 매핑
BREED_MAP = {
    "포메라니안": "pomeranian",
    "골든리트리버": "golden retriever",
    "시바견": "shiba inu",
    "푸들": "poodle",
    "웰시코기": "corgi",
    "말티즈": "maltese",
    "비숑프리제": "bichon frise",
    "치와와": "chihuahua",
    "닥스훈트": "dachshund",
    "진돗개": "jindo dog",
}

COLOR_MAP = {
    "흰색": "white",
    "검정": "black",
    "갈색": "brown",
    "베이지": "cream beige",
    "회색": "gray",
    "황금색": "golden",
    "검흰": "black and white",
    "갈흰": "brown and white",
}

PERSONALITY_MAP = {
    "활발한": "energetic pose, big happy smile, excited expression",
    "차분한": "calm sitting pose, gentle smile, peaceful expression",
    "장난꾸러기": "playful pose, mischievous grin, tilted head",
    "귀여운": "adorable pose, sparkling big eyes, sweet smile",
    "당당한": "confident standing pose, proud expression, head up",
    "소심한": "shy pose, small smile, looking up cutely",
}


@app.route("/generate", methods=["POST"])
def generate():
    breed_kr = request.form.get("breed", "포메라니안")
    color_kr = request.form.get("color", "흰색")
    personality_kr = request.form.get("personality", "활발한")
    seed = int(request.form.get("seed", 42))

    breed = BREED_MAP.get(breed_kr, breed_kr)
    color = COLOR_MAP.get(color_kr, color_kr)
    personality = PERSONALITY_MAP.get(personality_kr, personality_kr)

    prompt = (
        f"cute {color} {breed} dog mascot character, {personality}, "
        "flat vector illustration, chibi proportions, "
        "big round eyes, thick black outline, flat colors, "
        "full body, entire body inside frame, "
        "isolated on plain white background, no ground, no shadow, "
        "clean digital illustration"
    )
    negative_prompt = (
        "realistic, photo, photorealistic, 3d render, real fur texture, "
        "cropped, cut off, out of frame, close up, "
        "ground, floor, shadow, object, "
        "text, watermark, logo, deformed, extra limbs, blurry, low quality, ugly"
    )

    generator = torch.Generator(device="cuda").manual_seed(seed)
    img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=generator,
    ).images[0]

    final = remove(img, session=bg_session)

    buf = io.BytesIO()
    final.save(buf, format="PNG")
    buf.seek(0)
    return send_file(buf, mimetype="image/png")


ngrok.set_auth_token("")
public_url = ngrok.connect(5000)
print("API URL:", public_url)
from flask import Flask, request, send_file
from flask_cors import CORS
from pyngrok import ngrok
import io, torch
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from rembg import remove, new_session

app = Flask(__name__)
CORS(app)

# LoRA 없이 base 모델만 (특정 개체에 묶이지 않도록)
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
bg_session = new_session("u2net")

# 한글 입력을 영어 프롬프트로 매핑
BREED_MAP = {
    "포메라니안": "pomeranian",
    "골든리트리버": "golden retriever",
    "시바견": "shiba inu",
    "푸들": "poodle",
    "웰시코기": "corgi",
    "말티즈": "maltese",
    "비숑프리제": "bichon frise",
    "치와와": "chihuahua",
    "닥스훈트": "dachshund",
    "진돗개": "jindo dog",
}

COLOR_MAP = {
    "흰색": "white",
    "검정": "black",
    "갈색": "brown",
    "베이지": "cream beige",
    "회색": "gray",
    "황금색": "golden",
    "검흰": "black and white",
    "갈흰": "brown and white",
}

PERSONALITY_MAP = {
    "활발한": "energetic pose, big happy smile, excited expression",
    "차분한": "calm sitting pose, gentle smile, peaceful expression",
    "장난꾸러기": "playful pose, mischievous grin, tilted head",
    "귀여운": "adorable pose, sparkling big eyes, sweet smile",
    "당당한": "confident standing pose, proud expression, head up",
    "소심한": "shy pose, small smile, looking up cutely",
}


@app.route("/generate", methods=["POST"])
def generate():
    breed_kr = request.form.get("breed", "포메라니안")
    color_kr = request.form.get("color", "흰색")
    personality_kr = request.form.get("personality", "활발한")
    seed = int(request.form.get("seed", 42))

    breed = BREED_MAP.get(breed_kr, breed_kr)
    color = COLOR_MAP.get(color_kr, color_kr)
    personality = PERSONALITY_MAP.get(personality_kr, personality_kr)

    prompt = (
        f"cute {color} {breed} dog mascot character, {personality}, "
        "flat vector illustration, chibi proportions, "
        "big round eyes, thick black outline, flat colors, "
        "full body, entire body inside frame, "
        "isolated on plain white background, no ground, no shadow, "
        "clean digital illustration"
    )
    negative_prompt = (
        "realistic, photo, photorealistic, 3d render, real fur texture, "
        "cropped, cut off, out of frame, close up, "
        "ground, floor, shadow, object, "
        "text, watermark, logo, deformed, extra limbs, blurry, low quality, ugly"
    )

    generator = torch.Generator(device="cuda").manual_seed(seed)
    img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=generator,
    ).images[0]

    final = remove(img, session=bg_session)

    buf = io.BytesIO()
    final.save(buf, format="PNG")
    buf.seek(0)
    return send_file(buf, mimetype="image/png")


ngrok.set_auth_token("3IuTYphCzh7b5C45QsHeLqqnRF2_5gVfHjeKEmHecPZYTG5ct")
public_url = ngrok.connect(5000)
print("API URL:", public_url)
app.run(port=5000, threaded=True)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

...

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

ERROR:pyngrok.process.ngrok:t=2026-09-06T12:30:59+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-09-06T12:30:59+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
import requests
from PIL import Image
from io import BytesIO

res = requests.post(
    "http://127.0.0.1:5000/generate",
    data={"breed": "포메라니안", "color": "흰색", "personality": "활발한", "seed": 42},
)
if res.status_code == 200:
    display(Image.open(BytesIO(res.content)))
else:
    print(res.status_code, res.text)